In [ ]:
"""Minimal example: load verified eye data (degrees) and accelerometer data for a single block.

Usage:
- Set `block_path` to the block folder you want (e.g. .../PV_208/2025_12_14/block_019).
- Run this cell; it will:
  * Initialize a BlockSync object
  * Load right/left eye CSVs exported by the pipeline (degree-based)
  * Initialize `oe_rec`
  * Load accelerometer data via OERecording.get_accel_data
"""

from pathlib import Path
import sys

import numpy as np
import pandas as pd

# -------------------------------------------------------------------------
# Imports so this works when run from the repo root
# -------------------------------------------------------------------------
# Adjust PYTHONPATH so `eye_tracking_system_tools` is importable
REPO_SRC = Path.cwd() / "src"  # assumes you start the notebook in repo root
if str(REPO_SRC) not in sys.path:
    sys.path.insert(0, str(REPO_SRC))

from eye_tracking_system_tools.preprocessing.BlockSync_class import BlockSync

# -------------------------------------------------------------------------
# 1) Point to a block folder and build the BlockSync object
# -------------------------------------------------------------------------

# TODO: change this to your actual block folder
block_path = Path(r"D:\sample_data_for_eye_repo\PV_208\2025_12_14\block_019")  # <- change me

if not block_path.is_dir():
    raise FileNotFoundError(f"Not a directory: {block_path}")

# Match logic from explore_analog_channels.build_block_from_path
animal_call = block_path.parent.parent.name       # e.g. PV_208
experiment_date = block_path.parent.name          # e.g. 2025_12_14
block_num = block_path.name.replace("block_", "") # e.g. "019"
path_to_animal_folder = str(block_path.parent.parent.parent)

block = BlockSync(
    animal_call=animal_call,
    experiment_date=experiment_date,
    block_num=block_num,
    path_to_animal_folder=path_to_animal_folder,
)

print(block)
print(f"Analysis path: {block.analysis_path}")

# -------------------------------------------------------------------------
# 2) Load verified degree-based eye data from CSVs
#    (exported by export_eye_data_2d in block_sync_core)
# -------------------------------------------------------------------------

right_eye_csv = Path(block.analysis_path) / "right_eye_data_degrees_raw_verified.csv"
left_eye_csv = Path(block.analysis_path) / "left_eye_data_degrees_raw_verified.csv"

if not right_eye_csv.is_file() or not left_eye_csv.is_file():
    raise FileNotFoundError(
        "Eye CSVs not found.\n"
        f"  Expected: {right_eye_csv}\n"
        f"            {left_eye_csv}\n"
        "Make sure you've run the eye pipeline and export_eye_data_2d for this block."
    )

block.right_eye_data = pd.read_csv(right_eye_csv)
block.left_eye_data = pd.read_csv(left_eye_csv)

print("Right eye columns:", list(right_eye_df.columns))
print("Left eye columns:", list(left_eye_df.columns))

# -------------------------------------------------------------------------
# 3) Initialize oe_rec and load accelerometer data (AUX channels)
# -------------------------------------------------------------------------

oe_rec = block.oe_rec
if oe_rec is None:
    raise RuntimeError("Block has no Open Ephys recording (block.oe_rec is None).")

start_time_ms = np.array([[0.0]])          # single window starting at t=0
window_ms = oe_rec.recordingDuration_ms    # full recording

# get_accel_data: returns values in mV when convert_microvolts=True
accel_data, accel_timestamps = oe_rec.get_accel_data(
    channels=[],
    start_time_ms=start_time_ms,
    window_ms=window_ms,
    convert_microvolts=True,
    return_timestamps=True,
)

print("Accelerometer data shape (n_ch, n_windows, n_samples):", accel_data.shape)
print("Accelerometer timestamps shape (n_windows, n_samples):", accel_timestamps.shape)

# At this point you have:
# - right_eye_df / left_eye_df: verified, degree-based eye data
# - accel_data / accel_timestamps: accelerometer traces to align with eye data

In [ ]:
from bokeh.io import output_notebook, show, output_file
from bokeh.plotting import figure
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, HoverTool

output_notebook()

def plot_eye_and_accel(
    right_eye_df,
    left_eye_df,
    accel_data,
    accel_timestamps,
    *,
    # Eye dataframe column names (edit these to match your CSVs)
    time_col_eye="time_ms",          # or e.g. "t_ms"
    right_theta_col="theta_deg",     # right eye horizontal angle (deg)
    right_phi_col="phi_deg",         # right eye vertical angle (deg)
    left_theta_col="theta_deg",      # left eye horizontal angle (deg)
    left_phi_col="phi_deg",          # left eye vertical angle (deg)
    right_pupil_col="pupil_diam",    # right pupil diameter
    left_pupil_col="pupil_diam",     # left pupil diameter
    # Accelerometer channel indices (1-based indices into accel_data)
    accel_channels_to_plot=None,
):
    """
    Plot left/right eye angles, pupil diameter, and accelerometer traces
    in an interactive Bokeh layout inside the notebook.
    """
    # ------------------------------
    # Prepare eye data
    # ------------------------------
    if time_col_eye not in right_eye_df.columns:
        raise KeyError(f"{time_col_eye} not found in right_eye_df columns: {right_eye_df.columns}")
    if time_col_eye not in left_eye_df.columns:
        raise KeyError(f"{time_col_eye} not found in left_eye_df columns: {left_eye_df.columns}")

    t_eye_r = right_eye_df[time_col_eye].values.astype(float) / 1000.0  # convert ms -> s if needed
    t_eye_l = left_eye_df[time_col_eye].values.astype(float) / 1000.0

    # Angle columns may differ between R and L; allow separate names
    def _check(col, df, label):
        if col not in df.columns:
            raise KeyError(f"{col} not found in {label} columns: {df.columns}")

    for col, df, label in [
        (right_theta_col, right_eye_df, "right_eye_df"),
        (right_phi_col, right_eye_df, "right_eye_df"),
        (left_theta_col, left_eye_df, "left_eye_df"),
        (left_phi_col, left_eye_df, "left_eye_df"),
        (right_pupil_col, right_eye_df, "right_eye_df"),
        (left_pupil_col, left_eye_df, "left_eye_df"),
    ]:
        _check(col, df, label)

    r_theta = right_eye_df[right_theta_col].values
    r_phi = right_eye_df[right_phi_col].values
    l_theta = left_eye_df[left_theta_col].values
    l_phi = left_eye_df[left_phi_col].values

    r_pupil = right_eye_df[right_pupil_col].values
    l_pupil = left_eye_df[left_pupil_col].values

    # ------------------------------
    # Prepare accelerometer data
    # ------------------------------
    # accel_data: (n_ch, n_windows, n_samples) – you used a single window, so squeeze window axis.
    accel_data = np.asarray(accel_data)
    accel_timestamps = np.asarray(accel_timestamps)

    if accel_data.ndim != 3 or accel_data.shape[1] != 1:
        raise ValueError(f"Expected accel_data shape (n_ch, 1, n_samples); got {accel_data.shape}")

    n_ch, _, n_samples = accel_data.shape
    accel_traces = accel_data[:, 0, :]          # (n_ch, n_samples)
    accel_time_s = accel_timestamps[0, :] / 1000.0  # ms -> s

    if accel_channels_to_plot is None:
        accel_channels_to_plot = list(range(1, n_ch + 1))  # 1-based

    # Convert to 0-based indices internally
    accel_indices_0b = [c - 1 for c in accel_channels_to_plot if 1 <= c <= n_ch]
    if not accel_indices_0b:
        raise ValueError(f"No valid accel channels to plot from {accel_channels_to_plot} with n_ch={n_ch}")

    # ------------------------------
    # Bokeh: Eye angles figure
    # ------------------------------
    src_eye_r = ColumnDataSource(
        data=dict(
            t=t_eye_r,
            r_theta=r_theta,
            r_phi=r_phi,
            r_pupil=r_pupil,
        )
    )
    src_eye_l = ColumnDataSource(
        data=dict(
            t=t_eye_l,
            l_theta=l_theta,
            l_phi=l_phi,
            l_pupil=l_pupil,
        )
    )

    p_angles = figure(
        width=900,
        height=300,
        title="Eye angles (degrees)",
        x_axis_label="Time (s)",
        y_axis_label="Angle (deg)",
        tools="pan,wheel_zoom,box_zoom,reset,save,hover",
        active_scroll="wheel_zoom",
    )

    # Right eye
    p_angles.line("t", "r_theta", source=src_eye_r, line_color="red", line_width=1.5, legend_label="R θ")
    p_angles.line("t", "r_phi", source=src_eye_r, line_color="orange", line_width=1.5, legend_label="R φ")

    # Left eye
    p_angles.line("t", "l_theta", source=src_eye_l, line_color="blue", line_width=1.5, legend_label="L θ")
    p_angles.line("t", "l_phi", source=src_eye_l, line_color="green", line_width=1.5, legend_label="L φ")

    p_angles.legend.click_policy = "hide"
    p_angles.select_one(HoverTool).tooltips = [
        ("time (s)", "@t{0.000}"),
        ("R θ", "@r_theta{0.0}"),
        ("R φ", "@r_phi{0.0}"),
        ("L θ", "@l_theta{0.0}"),
        ("L φ", "@l_phi{0.0}"),
    ]

    # ------------------------------
    # Bokeh: Pupil diameters figure
    # ------------------------------
    p_pupil = figure(
        width=900,
        height=200,
        x_range=p_angles.x_range,  # link x-axis to angles figure
        title="Pupil diameter",
        x_axis_label="Time (s)",
        y_axis_label="Diameter (a.u.)",
        tools="pan,wheel_zoom,box_zoom,reset,save,hover",
        active_scroll="wheel_zoom",
    )

    p_pupil.line("t", "r_pupil", source=src_eye_r, line_color="purple", line_width=1.5, legend_label="R pupil")
    p_pupil.line("t", "l_pupil", source=src_eye_l, line_color="black", line_width=1.5, legend_label="L pupil")

    p_pupil.legend.click_policy = "hide"
    p_pupil.select_one(HoverTool).tooltips = [
        ("time (s)", "@t{0.000}"),
        ("R pupil", "@r_pupil{0.000}"),
        ("L pupil", "@l_pupil{0.000}"),
    ]

    # ------------------------------
    # Bokeh: Accelerometer figure
    # ------------------------------
    accel_src = ColumnDataSource(
        data=dict(
            t=accel_time_s,
            **{
                f"ch{idx+1}": accel_traces[idx, :]
                for idx in accel_indices_0b
            }
        )
    )

    p_accel = figure(
        width=900,
        height=250,
        x_range=p_angles.x_range,  # link x-axis
        title="Accelerometer traces (mV)",
        x_axis_label="Time (s)",
        y_axis_label="Accel (mV)",
        tools="pan,wheel_zoom,box_zoom,reset,save,hover",
        active_scroll="wheel_zoom",
    )

    palette = ["firebrick", "navy", "olive", "goldenrod", "teal", "darkmagenta"]
    for j, idx in enumerate(accel_indices_0b):
        ch_name = f"ch{idx+1}"
        p_accel.line(
            "t",
            ch_name,
            source=accel_src,
            line_width=1.2,
            color=palette[j % len(palette)],
            legend_label=f"Accel ch {idx+1}",
        )

    p_accel.legend.click_policy = "hide"
    p_accel.select_one(HoverTool).tooltips = [("time (s)", "@t{0.000}")] + [
        (f"ch {idx+1}", f"@ch{idx+1}{{0.000}}") for idx in accel_indices_0b
    ]

    # ------------------------------
    # Show all together
    # ------------------------------
    output_file("eye_and_accel.html")
    show(column(p_angles, p_pupil, p_accel))


# Example call (add after the function definition, still in this cell):
plot_eye_and_accel(
    right_eye_df=right_eye_df,
    left_eye_df=left_eye_df,
    accel_data=accel_data,
    accel_timestamps=accel_timestamps,
    time_col_eye="ms_axis",          # change if your CSV uses a different name
    right_theta_col="k_theta",     # change to your actual column names
    right_phi_col="k_phi",
    left_theta_col="k_theta",
    left_phi_col="k_phi",
    right_pupil_col="major_ax",
    left_pupil_col="major_ax",
    accel_channels_to_plot=[1,2,3],     # or e.g. [1, 2, 3]
)